# Exploration des données industrielles (MVTec AD)

Objectif : comprendre la structure du dataset, vérifier les splits, visualiser
des exemples normaux et anormaux, et inspecter les distributions.

> Ce notebook **explore** ; il ne remplace pas le package `src/`. Toute la
> logique réutilisable vit dans `anomaly_detection.data`.

Si le vrai MVTec AD n'est pas encore téléchargé, on se rabat automatiquement
sur une **catégorie synthétique** pour que le notebook tourne quand même.

In [ ]:
import sys
from pathlib import Path

# Rend le package importable sans installation editable.
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt

from anomaly_detection.data.preprocessing import load_image
from anomaly_detection.data.splits import create_splits
from anomaly_detection.data.synthetic import generate_synthetic_mvtec

In [ ]:
# Choix du dataset : vrai MVTec si présent, sinon synthétique.
DATA_ROOT = ROOT / "data" / "raw" / "mvtec_ad"
CATEGORY = "bottle"

if not (DATA_ROOT / CATEGORY / "train" / "good").is_dir():
    print("MVTec non trouve -> generation d'une categorie synthetique.")
    generate_synthetic_mvtec(DATA_ROOT, category="synthetic", n_train_good=12)
    CATEGORY = "synthetic"

print("Categorie utilisee :", CATEGORY)

In [ ]:
# Splits reproductibles et comptages par jeu.
splits = create_splits(DATA_ROOT, CATEGORY, val_fraction=0.2, seed=42)
for name, entries in splits.items():
    n_normal = sum(e["label"] == 0 for e in entries)
    n_anom = sum(e["label"] == 1 for e in entries)
    print(
        f"{name:5s} : {len(entries):4d} images  (normal={n_normal}, anomalie={n_anom})"
    )

In [ ]:
# Distribution des types de defauts dans le jeu de test.
from collections import Counter

defects = Counter(e["defect_type"] for e in splits["test"])
plt.figure(figsize=(7, 3))
plt.bar(defects.keys(), defects.values())
plt.title(f"Types dans le test - {CATEGORY}")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation : une image normale vs une anormale (+ son masque).
normal = next(e for e in splits["test"] if e["label"] == 0)
anomaly = next(e for e in splits["test"] if e["label"] == 1)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(load_image(DATA_ROOT / normal["image"]))
axes[0].set_title("Normal")
axes[1].imshow(load_image(DATA_ROOT / anomaly["image"]))
axes[1].set_title(f"Anomalie ({anomaly['defect_type']})")
if anomaly["mask"]:
    axes[2].imshow(load_image(DATA_ROOT / anomaly["mask"]), cmap="gray")
    axes[2].set_title("Masque (verite terrain)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Inspection des dimensions des images normales d'entrainement.
sizes = [load_image(DATA_ROOT / e["image"]).size for e in splits["train"]]
widths, heights = zip(*sizes)
print("Largeurs uniques :", sorted(set(widths)))
print("Hauteurs uniques :", sorted(set(heights)))
print("-> justifie le redimensionnement a image_size dans les transforms.")

## Conclusions

- Le jeu **train** ne contient que des normales (cadre *one-class*).
- Le jeu **test** melange normales et anomalies, avec masques pixel pour ces
  dernieres -> metriques pixel-level possibles sur ce domaine.
- Les images ont une taille homogene par categorie, mais on redimensionne
  quand meme a `image_size` pour un pipeline uniforme entre domaines.

Prochaine etape (Phase 2) : entrainer un autoencodeur sur les normales.